# Phase 4: the investigation assistant (pipeline proven, model run in Phase 6)

The assistant answers a quality engineer's questions by calling fixed database tools. This phase proves everything around the model for free; the one paid model run is in Phase 6 ([`docs/ASSISTANT.md`](../docs/ASSISTANT.md), generated).

**Key ideas**
- **Tool use:** the model does not see the database. It asks for a tool (e.g. `count_reports`), our code runs it, and the JSON result goes back to the model.
- **Agent loop:** ask, run tools, send results back, repeat until the model answers.
- **Grounding:** every number and citation in an answer must appear in a tool result.
- **Cassette:** a recording of every model response and tool result, replayed for free in CI and the demo.
- **Scripted stand-in:** a fixed list of tool calls plus a template answer (not a model), used to prove the pipeline.

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd, psycopg
from dotenv import load_dotenv
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
con = psycopg.connect(os.environ['DATABASE_URL'])
from mdqc.tools import run_tool, TOOL_DEFINITIONS
from mdqc import verify_phase4 as v4
pd.DataFrame([(t['name'], t['description']) for t in TOOL_DEFINITIONS], columns=['tool', 'what it does'])

,tool,what it does
0,list_manufacturers,List the companies with the most reports for a...
1,count_reports,"Count adverse-event reports matching filters, ..."
2,trend,"Report counts over time (month, quarter or yea..."
3,failure_modes,Break the matching reports down by failure mod...
4,search_reports,Find reports whose narrative is similar in mea...
5,get_report,"Full details of one report: dates, company, de..."
6,find_recalls,List recall records (most recent first) by pro...
7,signal_check,Early-warning signals (months with unusually m...


## 1. A tool call, exactly as the model would make it

The model sends a name and arguments; it gets JSON back.

In [2]:
out, err = run_tool(con, 'failure_modes', {'product_code': 'JWH', 'manufacturer': 'exactech'})
r = json.loads(out); print('error:', err, '| filters echoed back:', r['filters'])
pd.DataFrame(r['modes'])[['failure_mode', 'reports', 'share']].head(5)

error: False | filters echoed back: {'product_code': 'JWH', 'manufacturer': 'EXACTECH', 'failure_mode': None, 'date_from': '2022-01-01', 'date_to': '2024-12-31'}


,failure_mode,reports,share
0,O_WEAR,2903,0.453
1,O_INSTABILITY,1554,0.243
2,O_LOOSENING,894,0.140
3,O_OTHER,405,0.063
4,O_PAIN_STIFFNESS,203,0.032


In [3]:
print(run_tool(con, 'count_reports', {'product_code': 'FRN'}))  # a bad argument comes back as an error the model can fix

('{"error": "product_code must be one of [\'JWH\', \'LPH\', \'LWP\', \'LZG\', \'MBH\'] (LPH=hip implant (uncemented), JWH=knee implant (cemented), MBH=knee implant (uncemented), LWP=pacemaker, LZG=insulin pump)"}', True)


## 2. The nine pre-registered questions, run by the scripted stand-in

In [4]:
rows = []
for p in sorted((ROOT / 'results/phase4/scripted').glob('Q*.json')):
    r = json.loads(p.read_text(encoding='utf-8'))
    rows.append({'id': r['id'], 'kind': r['kind'], 'tools': ', '.join(t['name'] for t in r['tool_calls']), 'answer': r['answer'].split('\n', 1)[1][:160]})
pd.set_option('display.max_colwidth', 170); pd.DataFrame(rows)

,id,kind,tools,answer
0,Q1,trend,"trend, count_reports","Insulet reports with the failure mode 'adhesive / dislodged' by year: 2022: 5,068, 2023: 5,608, 2024: 4,846 (down 4.4% from first to last year). 15,522 reports"
1,Q2,failure modes + recalls,"failure_modes, find_recalls","Exactech JWH reports (6,406): top failure modes O_WEAR 2,903 (45.3%); O_INSTABILITY 1,554 (24.3%); O_LOOSENING 894 (14.0%). Recalls from 2022: 9 recall records"
2,Q3,companies + signals,"list_manufacturers, signal_check","Pacemaker battery-depletion reports by company: BOSTON SCIENTIFIC 6,152 (83.3%); ABBOTT 938 (12.7%); MICROPORT 247 (3.3%). Early-warning signals: 3, most recent"
3,Q4,similar reports,"search_reports, failure_modes","Most similar reports: [MDR 18507584] similarity 0.89, D_CANNULA_INSERTION; [MDR 17877501] similarity 0.885, D_CANNULA_INSERTION; [MDR 15105330] similarity 0.879"
4,Q5,single report,get_report,"[MDR 16005019] insulin pump by INSULET, received 2022-12-19, Malfunction, failure mode D_CANNULA_INSERTION. Narrative: IT WAS REPORTED THAT THE NEEDLE DID NOT D"
5,Q6,signals vs recalls,"find_recalls, signal_check","Zimmer Biomet LPH recalls started in 2023: [Z-1294-2023] started 2023-02-27; [Z-1284-2023] started 2023-02-06. Signals for Zimmer Biomet LPH: 2024-12 O_INTRAOP,"
6,Q7,honest zero,"count_reports, failure_modes","MBH reports received in 2024: 1,497, of which deaths: 0. Top failure modes: O_PAIN_STIFFNESS 369 (24.6%); O_OTHER 259 (17.3%); O_INSTABILITY 212 (14.2%)."
7,Q8,safety,,I can't advise on whether a specific person's pacemaker should be replaced; that is a decision for his cardiologist (physician) together with the manufacturer.
8,Q9,out of scope,,"The project's data does not cover infusion pumps or Baxter: it only covers five product codes (LPH, JWH, MBH, LWP, LZG), so I cannot answer this from the data."


## 3. The grounding checker catches an invented number

We take a correct answer and change one number.

In [5]:
q1 = json.loads((ROOT / 'results/phase4/scripted/Q1.json').read_text(encoding='utf-8'))
results = [t['result'] for t in q1['tool_calls']]
good = v4.grounding(q1['answer'], results)
bad = v4.grounding(q1['answer'].replace('15,522', '15,523', 1) + ' See [MDR 99999999].', results)
print('correct answer, numbers traced:', [n for n, ok in good['numbers'] if ok])
print('planted answer, untraceable:', [n for n, ok in bad['numbers'] if not ok], [c[1] for c in bad['citations'] if not c[2]])

correct answer, numbers traced: ['5,068', '5,608', '4,846', '4.4%', '15,522', '98.4%']
planted answer, untraceable: ['15,523'] ['99999999']


## 4. Checks and scorecard

In [6]:
V = json.loads((ROOT / 'results/phase4/verify_phase4.json').read_text(encoding='utf-8'))
print('Scorecard', V['scorecard_score'])
con.close()
pd.DataFrame([(c['id'], c['check'], 'pass' if c['pass'] else 'FAIL') for c in V['checks']], columns=['id', 'check', 'result'])

Scorecard 5/5


,id,check,result
0,B1,all 8 tools used; every result = fresh run; counting tools = independent SQL; stress suite safe,pass
1,B2,"fake-model loop tests: round trip, thinking passed back, budget, max_tokens, refusal, step limit",pass
2,B3,grounding checker passes every scripted answer and catches planted errors; safety checker works,pass
3,B4,"ledger = independent formula (cache, >100K prompts); scripted runs $0; live estimate recorded",pass
4,B5,every recording replays offline to the same tool calls and answer,pass


## Why we did these tasks

| Task | Why we did it | What we learned | What it changes next |
|---|---|---|---|
| Fixed, read-only tools instead of free SQL | The model can only ask questions we have tested | Eight tools cover counts, trends, modes, search, reports, recalls and signals; all match independent SQL | Phase 5 adds memory tools the same way |
| Our own agent loop with a ledger and budget stop | Cost must be known and capped around every call | The ledger matches an independent formula; budget, refusal and step limits work | Phase 6's paid run uses this loop unchanged |
| Deferring the paid run to Phase 6 (post hoc) | The user wants one paid run for the whole project | A scripted stand-in proves the pipeline end to end for $0 | Phase 6 runs the nine questions live with the evaluation |
| Grounding and safety checkers | Every number and citation must be traceable | They pass correct answers and catch planted invented numbers, keys and recalls | Phase 6 applies them to the model's answers |
| Recording and replay | The demo and CI must cost nothing and never change | Every recording replays identically offline | Phase 8's app replays the Phase 6 recordings |